In [ ]:
import sys
sys.path.append("../models")
from SVD import SVDModel

In [2]:
import pandas as pd

movies = pd.read_csv('../data/sample/movies.csv')
ratings = pd.read_csv('../data/sample/ratings.csv')
tags = pd.read_csv('../data/sample/tags.csv')

In [65]:
import sys
import os

# Thay vì dùng __file__, Jupyter sử dụng os.getcwd() để lấy thư mục hiện tại
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))

if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import torch
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
from data.dataloader import MovieLensDataLoader
from SVD import SVDModel
loader = MovieLensDataLoader(data_dir="../data/sample")
data_bundle = loader.load()

movies_df = data_bundle.movies
ratings_df = data_bundle.ratings

num_users = ratings_df['userId'].nunique()
num_items = ratings_df['movieId'].nunique()
global_mean = ratings_df['rating'].mean()

user_to_idx = {uid: i for i, uid in enumerate(ratings_df['userId'].unique())}
item_to_idx = {mid: i for i, mid in enumerate(ratings_df['movieId'].unique())}

train_df, val_df, test_df = loader.train_val_test_split(ratings_df, val_ratio=0.1, test_ratio=0.1)

def df_to_tensors(df, user_to_idx, item_to_idx):
    df = df[df['userId'].isin(user_to_idx) & df['movieId'].isin(item_to_idx)]
    
    u_tensor = torch.tensor([user_to_idx[uid] for uid in df['userId']], dtype=torch.long)
    i_tensor = torch.tensor([item_to_idx[mid] for mid in df['movieId']], dtype=torch.long)
    r_tensor = torch.tensor(df['rating'].values, dtype=torch.float32)
    return u_tensor, i_tensor, r_tensor

train_user, train_item, train_rating = df_to_tensors(train_df, user_to_idx, item_to_idx)
val_user, val_item, val_rating = df_to_tensors(val_df, user_to_idx, item_to_idx)

train_dataset = TensorDataset(train_user, train_item, train_rating)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = SVDModel(num_users, num_items, embedding_dim=2, global_mean=global_mean).to(device)
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.2) # Thêm weight_decay để tránh overfit

epochs = 4
for epoch in range(epochs):
    model.train()
    total_train_loss = 0.0
    for batch_user, batch_item, batch_rating in train_loader:
        batch_user, batch_item, batch_rating = batch_user.to(device), batch_item.to(device), batch_rating.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_user, batch_item)
        loss = criterion(preds, batch_rating)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * batch_user.size(0)
        
    avg_train_loss = total_train_loss / len(train_dataset)
    
    model.eval()
    with torch.no_grad():
        val_user, val_item, val_rating = val_user.to(device), val_item.to(device), val_rating.to(device)
        val_preds = model(val_user, val_item)
        val_loss = criterion(val_preds, val_rating).item()
        
    print(f"Epoch {epoch+1:02d}/{epochs} -> Train Loss (MSE): {avg_train_loss:.4f} | Val Loss (MSE): {val_loss:.4f}")

torch.save({
    'model_state_dict': model.state_dict(),
    'user_to_idx': user_to_idx,
    'item_to_idx': item_to_idx
}, '../artifacts/svd_model_v2.pth')

Epoch 01/4 -> Train Loss (MSE): 0.1608 | Val Loss (MSE): 0.2746
Epoch 02/4 -> Train Loss (MSE): 0.1601 | Val Loss (MSE): 0.2754
Epoch 03/4 -> Train Loss (MSE): 0.1594 | Val Loss (MSE): 0.2761
Epoch 04/4 -> Train Loss (MSE): 0.1587 | Val Loss (MSE): 0.2767


In [66]:
def recommend_movies(model, user_id, ratings, movies, user_to_idx, item_to_idx, top_k=10, device='cpu'):
    model.eval()
    with torch.no_grad():
        # Kiểm tra user_id có trong dataset không
        if user_id not in user_to_idx:
            raise ValueError(f"User {user_id} not found in training data")
        
        # Lấy index user
        u_idx = user_to_idx[user_id]
        num_items = len(item_to_idx)
        
        # Tạo tensor tất cả item cho user này
        user_tensor = torch.tensor([u_idx] * num_items, dtype=torch.long).to(device)
        item_tensor = torch.tensor(list(range(num_items)), dtype=torch.long).to(device)
        
        # Dự đoán rating
        preds = model(user_tensor, item_tensor)
        
        # Chuyển về CPU và pandas Series
        preds = preds.cpu().numpy()
        item_idx_to_id = {idx: mid for mid, idx in item_to_idx.items()}
        pred_series = pd.Series(preds, index=[item_idx_to_id[i] for i in range(num_items)])
        
        # Loại bỏ những phim user đã đánh giá
        watched_items = ratings[ratings['userId'] == user_id]['movieId'].tolist()
        pred_series = pred_series.drop(watched_items, errors='ignore')
        
        # Lấy top_k phim dự đoán rating cao nhất
        top_movies_ids = pred_series.sort_values(ascending=False).head(top_k).index.tolist()
        
        # Lấy tên phim
        top_movies = movies[movies['movieId'].isin(top_movies_ids)][['movieId','title']]
        top_movies = top_movies.set_index('movieId').loc[top_movies_ids]  # giữ thứ tự top k
        
        return top_movies

top_movies = recommend_movies(model, user_id=101, ratings=ratings, movies=movies,
                              user_to_idx=user_to_idx, item_to_idx=item_to_idx,
                              top_k=10, device=device)

print(top_movies)

                                          title
movieId                                        
17                            The Matrix (1999)
14       Star Wars Episode IV A New Hope (1977)
3                                   Heat (1995)
4                                Sabrina (1995)
18                            Fight Club (1999)
8                     Usual Suspects The (1995)
7                                  Seven (1995)
21                             Inception (2010)
11                             Apollo 13 (1995)
13                           Taxi Driver (1976)
